In [1]:
import pandas as pd
import gzip
import json
from pm4py.utils import format_dataframe
from pm4py import write_xes
from tqdm.notebook import tqdm

In [2]:
def binet_to_df(path):
    with gzip.open(path, "r") as f:
        data = f.read()
        j = json.loads(data.decode('utf-8'))
    
    res_list = []
    
    for case in j['cases']:
        trace = pd.DataFrame.from_dict(case['events'])
        trace['anomaly'] = case['attributes']['label'] if isinstance(case['attributes']['label'], str) else case['attributes']['label']['anomaly']
        trace['trace_id'] = case['id']
        res_list.append(trace)
    
    if res_list:
        res = pd.concat(res_list, ignore_index=True)
        res = pd.concat([res.drop(['attributes'], axis=1), res['attributes'].apply(pd.Series)], axis=1)
    else:
        res = pd.DataFrame()
    
    return res

from datetime import datetime, timedelta

def assign_sequential_timestamps(df, start_time=None, step_minutes=10, duration_minutes=5):
    if start_time is None:
        start_time = datetime.now()

    df = df.copy()
    timestamps = []
    timestamps_end = []

    for trace_id, group in df.groupby('trace_id'):
        base_time = start_time
        for _ in range(len(group)):
            timestamps.append(base_time)
            timestamps_end.append(base_time + timedelta(minutes=duration_minutes))
            base_time += timedelta(minutes=step_minutes)

    df['timestamp'] = timestamps
    df['timestamp_end'] = timestamps_end
    return df

def convert_to_pm4py_df(df):
    df = df.copy()
    df = df.rename(columns={'name': 'activity', 'trace_id':'case_id', 'user':'org:resource', 'anomaly': 'anomaly'})
    df = df.astype({'activity': str, 'anomaly': str, 'org:resource': str})
    df = format_dataframe(df, case_id='case_id',activity_key='activity', timestamp_key='timestamp')
    df = df.drop(['activity', 'timestamp', 'timestamp_end'], axis=1)
    return df

def convert_and_write_json_to_xes(path_to_json, path_to_xes):
    df = binet_to_df(path_to_json)
    df = assign_sequential_timestamps(df)
    df = convert_to_pm4py_df(df)
    write_xes(df, path_to_xes)

In [3]:
# dataset_names = ["medium", "small", "p2p", "paper"]
dataset_names = ["wide", "bpic13","bpic15","bpic17"]
for dataset_name in tqdm(dataset_names, desc="Converting datasets"):
    print(f"Exporting {dataset_name}")
    dataset_json_path = r".out/eventlogs/{}-0.3-1.json.gz".format(dataset_name)
    dataset_xes_path = r".out/eventlogs/{}-0.3-1.xes".format(dataset_name)
    convert_and_write_json_to_xes(dataset_json_path, dataset_xes_path)

Converting datasets:   0%|          | 0/4 [00:00<?, ?it/s]

Exporting wide


exporting log, completed traces ::   0%|          | 0/5000 [00:00<?, ?it/s]

Exporting bpic13


exporting log, completed traces ::   0%|          | 0/1487 [00:00<?, ?it/s]

Exporting bpic15


exporting log, completed traces ::   0%|          | 0/1199 [00:00<?, ?it/s]

Exporting bpic17


exporting log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]